# Solve
Uses an LLM to solve the math problems step-by-step

In [ ]:
import os
from typing import List, Dict, Tuple
import json
import re
from tqdm import tqdm

from dotenv import load_dotenv
from openai import OpenAI

import scipy.stats as st
import pandas as pd
import numpy as np

from src.datasets import EEDIDataset

from src.prompt_util import prompt_openai

In [ ]:
load_dotenv()

### Dataset

In [ ]:
data_folder = "eedi_data"
dataset = EEDIDataset(data_folder, n_limit=500)
print(f"We have {len(dataset)} questions")

### Prompting

In [ ]:
gpt_5_config = {
    "base_url": "https://api.openai.com/v1/",
    "api_key_var": "OPENAI_API_KEY",
    "model": "gpt-5.2-2025-12-11",
    "completion_kwargs": {
        "max_completion_tokens": 16*1024,
        "temperature": 0.0,
    }
}

In [ ]:
model_config = gpt_5_config
client = OpenAI(base_url=model_config["base_url"], api_key=os.environ.get(model_config["api_key_var"], None))

output_folder = os.path.join(data_folder, f"{model_config['model']}_temp_{model_config['completion_kwargs']['temperature']}")
os.makedirs(output_folder, exist_ok=True)

In [ ]:
def generate_expert_rt_solution(
    current_question: Dict[str, str],
    client: OpenAI, 
    model_config: Dict[str, str]
) -> str:
    """
        Generates correct step-by-step solutions with expert-student reasoning traces for a given math question, 
        giving the model a few in-context demonstrations as well
    """
    final_answer = current_question['Answer']
    question = current_question['Question']

    system_prompt = "You are a helpful assistant."

    user_prompt = f"""
Solve the math problem. Output format (exact):
[STEP-1]...
...
[STEP-N]...
[FINAL ANSWER]...

Constraints: no fluff; only one reasoning/ arithmetic step at a time; show key work. Final answer MUST equal:
\"{final_answer}\"

Problem:
\"{question}\
"""

    return prompt_openai(client, system_prompt, user_prompt, model_config)

In [ ]:
save_every_n_questions = 10
include_first_high_level_error = True

raw_expert_responses_by_datapointid = {}
for i in tqdm(list(range(len(dataset)))):
    # save checkpoints
    if i % save_every_n_questions == 0: 
        with open(os.path.join(output_folder, "raw_expert_responses_by_datapointid.json"), "w+") as f:
            json.dump(raw_expert_responses_by_datapointid, f)

    datapoint = dataset[i]
    problem = datapoint["Problem"]
    answer_expert = generate_expert_rt_solution(problem, client=client, model_config=model_config)

    raw_expert_responses_by_datapointid[str(i)] = answer_expert

# final save
with open(os.path.join(output_folder, "raw_expert_responses_by_datapointid.json"), "w+") as f:
    json.dump(raw_expert_responses_by_datapointid, f)

In [ ]:
with open(os.path.join(output_folder, "raw_expert_responses_by_datapointid.json"), "r+") as f:
    raw_expert_responses_by_datapointid = json.load(f)

In [ ]:
def parse_answer(raw_answer: str) -> Dict[str,str]:
    if "Final Answer:" not in raw_answer:
        # if the model does not explicitly give the final answer, we take the last line as the final answer
        reasoning_part = raw_answer
        final_answer_part = re.sub(r"^\d+\. ", "", raw_answer.split("\n")[-1], count=1)
    else:
        parts = raw_answer.split("Final Answer:")
        reasoning_part = parts[0]
        final_answer_part = parts[1]

    reasoning_text = reasoning_part.replace("Reasoning Trace (student-style):", "").replace("Reasoning Trace:", "").strip()
    final_answer = final_answer_part.strip()

    # Step 2: split reasoning trace into steps
    return {
        "Answer": final_answer,
        "Reasoning": reasoning_text
    }

expert_responses_by_datapointid = { dpid: parse_answer(raw_answer) for dpid,raw_answer in raw_expert_responses_by_datapointid.items() }

### Evaluation

In [ ]:
true_answer_by_datapointid = { 
     str(dpid): dataset[dpid]["Choices"]["CorrectAnswer"] 
     for dpid in range(len(dataset))
}

In [ ]:
def remove_formatting(text: str) -> str:
    """ Removes all formatting characters from a string """
    return "".join(text.split()).lower().replace("\\(", "").replace("\\)", "").replace("\(", "").replace("\)", "")

def correct(responses: List[str], correct_answer: str) -> int:
    """ Computes the number of responses that are equal to the correct answer """
    responses = [remove_formatting(r) for r in responses]
    correct_answer = remove_formatting(correct_answer)
    return sum(r == correct_answer for r in responses)

In [ ]:
def get_mean_and_ci(scores: np.ndarray, confidence: float = 0.95) -> Tuple[float, float]:
    """ Mean and confidence interval using t-distribution """
    mean = np.mean(scores)
    sem = st.sem(scores)
    n = len(scores)

    h = sem * st.t.ppf((1 + confidence) / 2, n - 1)
    return mean, h

In [ ]:
expert_results_by_datapointid = {}
for dpid in expert_responses_by_datapointid.keys():
    dpid = str(dpid)
    expert_responses = [expert_responses_by_datapointid[dpid]["Answer"]]
    correct_anser = true_answer_by_datapointid[dpid]

    expert_results_by_datapointid[dpid] = {
        "Correct": correct(expert_responses, correct_anser)
    }

# save results as CSV
expert_results = pd.DataFrame([{"Id": dpid, **result} for dpid,result in expert_results_by_datapointid.items()])
expert_results.to_csv(os.path.join(output_folder, "expert_results.csv"), index=False)

print("Expert:")
for c in expert_results.columns:
    if c == "Id": continue
    mean,confidence = get_mean_and_ci(expert_results[c])
    print(f"\t{c}: {mean:.2f} ± {confidence:.2f}")


### Manual Check / Correction
Check reasoning traces where the LLM did not arrive at the correct solution and correct where necessary (NOTE: we did not find any RTs that need correction, its just different output formatting)

In [ ]:
incorrect_dpids = [dpid for dpid,corr in expert_results_by_datapointid.items() if not corr["Correct"]]
print(len(incorrect_dpids))

In [ ]:
def print_question(datapointid: int):
    datapoint = dataset[int(datapointid)]
    print("Question: " + datapoint["Problem"]["Question"])
    print("Topic: " + datapoint["Problem"]["Topic"]) 
    print("Concept: " + datapoint["Problem"]["Concept"]) 
    print("Answer: " + datapoint["Problem"]["Answer"])
    distractor_info = {k:v for k,v in datapoint["Problem"].items() if k.startswith("Distractor")}
    for k in sorted(distractor_info, key=distractor_info.get): 
        print(f"{k}: {distractor_info[k]}")

for dpid in incorrect_dpids:
    response = expert_responses_by_datapointid[dpid]
    print(f"Id: {dpid}")
    print_question(dpid)
    print("\n")
    print("[MODEL RESPONSE]")
    print(response["Reasoning"])
    print(response["Answer"])
    print("\n\n")
    print("-"*40)
    print("\n\n")
    

### Export

In [ ]:
# map them to questionids, store as joined csv
df = pd.DataFrame([{"QuestionId": dataset.questionids[int(dpid)], "Reasoning": response["Reasoning"], "Answer": response["Answer"]} for dpid,response in expert_responses_by_datapointid.items()])
df.to_csv(os.path.join(output_folder, "eedi_data/correct_rts.csv"), index=False)